# UAS Bengkod 
### Danendra Farrel Haryo Wibowo 
### A11.2023.15025

# EDA 


# Import Library

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

# Konfigurasi tampilan
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Tema visualisasi
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

print('Library berhasil diimport!')

## Load Dataset

> [Sales and Marketing Dataset](https://www.kaggle.com/datasets/bhaskerpaul/sales-and-marketing-dataset)

In [ ]:

df = pd.read_csv('sales_marketing_dataset.csv')  # <-- SESUAIKAN NAMA FILE

print(f' Dataset berhasil dimuat!')
print(f'   Jumlah baris    : {df.shape[0]:,}')
print(f'   Jumlah kolom    : {df.shape[1]}')

---
## Tampilkan 5 Baris Pertama, Info Dataset, dan Statistik Deskriptif

### 1.1 Lima Baris Pertama Dataset

In [ ]:
print('=== 5 BARIS PERTAMA DATASET ===')
df.head()

### 1.2 Informasi Dataset

In [ ]:
print('=== INFORMASI DATASET ===')
df.info()

In [ ]:
print('=== RINGKASAN TIPE DATA ===')
dtype_summary = df.dtypes.value_counts().reset_index()
dtype_summary.columns = ['Tipe Data', 'Jumlah Kolom']
print(dtype_summary.to_string(index=False))

### 1.3 Statistik Deskriptif

In [ ]:
print('=== STATISTIK DESKRIPTIF — FITUR NUMERIK ===')
df.describe().T.style.background_gradient(cmap='Blues')

In [ ]:
print('=== STATISTIK DESKRIPTIF — FITUR KATEGORIKAL ===')
df.describe(include='object')


## Persentase Missing Value & Visualisasi Diagram Batang

In [ ]:
# Hitung jumlah dan persentase missing value
missing_count = df.isnull().sum()
missing_pct   = (missing_count / len(df)) * 100

missing_df = pd.DataFrame({
    'Kolom'            : missing_count.index,
    'Jumlah Missing'   : missing_count.values,
    'Persentase (%)'   : missing_pct.values
}).sort_values('Persentase (%)', ascending=False).reset_index(drop=True)

print('=== RINGKASAN MISSING VALUE ===')
print(f'Kolom tanpa missing value : {(missing_df["Jumlah Missing"] == 0).sum()}')
print(f'Kolom dengan missing value: {(missing_df["Jumlah Missing"] > 0).sum()}')
print()
print(missing_df[missing_df['Jumlah Missing'] > 0].to_string(index=False))

In [ ]:
# Filter hanya kolom yang memiliki missing value
missing_plot = missing_df[missing_df['Jumlah Missing'] > 0].copy()

if missing_plot.empty:
    print('Tidak ada missing value pada dataset ini.')
else:
    fig, ax = plt.subplots(figsize=(12, max(4, len(missing_plot) * 0.5 + 2)))

    bars = ax.barh(
        missing_plot['Kolom'],
        missing_plot['Persentase (%)'],
        color=sns.color_palette('Reds_r', len(missing_plot)),
        edgecolor='white',
        height=0.6
    )

    # Tambahkan label nilai
    for bar, val in zip(bars, missing_plot['Persentase (%)']):
        ax.text(
            bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f}%', va='center', fontsize=9, color='#333333'
        )

    ax.set_xlabel('Persentase Missing Value (%)', fontsize=11)
    ax.set_title('Persentase Missing Value per Kolom', fontsize=14, fontweight='bold', pad=12)
    ax.xaxis.set_major_formatter(mtick.PercentFormatter())
    ax.invert_yaxis()
    ax.spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.savefig('missing_value_chart.png', bbox_inches='tight')
    plt.show()
    print('\nGrafik missing value berhasil ditampilkan.')

---
## Distribusi Variabel Target 

In [ ]:
# Hitung distribusi Churn
churn_counts = df['churn'].value_counts().sort_index()
churn_pct    = df['churn'].value_counts(normalize=True).sort_index() * 100

churn_summary = pd.DataFrame({
    'Label'     : ['Tidak Churn (0)', 'Churn (1)'],
    'Jumlah'    : churn_counts.values,
    'Persentase': churn_pct.values
})

print('=== DISTRIBUSI VARIABEL TARGET — CHURN ===')
print(churn_summary.to_string(index=False))

ratio = churn_counts[0] / churn_counts[1]
print(f'\nRasio kelas (Tidak Churn : Churn) = {ratio:.2f} : 1')
if ratio > 1.5:
    print('Dataset kemungkinan mengalami class imbalance.')
else:
    print('Dataset relatif seimbang.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

labels   = ['Tidak Churn (0)', 'Churn (1)']
colors   = ['#4C9BE8', '#E8604C']
counts   = churn_counts.values
percents = churn_pct.values

# --- Plot kiri: Diagram batang jumlah ---
bars = axes[0].bar(labels, counts, color=colors, edgecolor='white', width=0.5)
for bar, count, pct in zip(bars, counts, percents):
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(counts) * 0.015,
        f'{count:,}\n({pct:.1f}%)',
        ha='center', va='bottom', fontsize=10, fontweight='bold'
    )
axes[0].set_title('Distribusi Kelas Churn (Jumlah)', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Jumlah Pelanggan')
axes[0].set_ylim(0, max(counts) * 1.2)
axes[0].spines[['top', 'right']].set_visible(False)

# --- Plot kanan: Diagram batang persentase ---
bars2 = axes[1].bar(labels, percents, color=colors, edgecolor='white', width=0.5)
for bar, pct in zip(bars2, percents):
    axes[1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(percents) * 0.015,
        f'{pct:.1f}%',
        ha='center', va='bottom', fontsize=11, fontweight='bold'
    )
axes[1].set_title('Distribusi Kelas Churn (%)', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Persentase (%)')
axes[1].set_ylim(0, max(percents) * 1.2)
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].spines[['top', 'right']].set_visible(False)

plt.suptitle('Keseimbangan Kelas — Variabel Target Churn', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('churn_distribution.png', bbox_inches='tight')
plt.show()
print('\n Grafik distribusi Churn berhasil ditampilkan.')

---
## Heatmap Korelasi Fitur Numerik

In [ ]:
# Pilih fitur numerik
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
print(f'Jumlah fitur numerik: {len(numeric_cols)}')
print('Daftar fitur numerik:')
for col in numeric_cols:
    print(f'  - {col}')

In [ ]:
# Hitung matriks korelasi
corr_matrix = df[numeric_cols].corr()

# Heatmap korelasi lengkap
fig, ax = plt.subplots(figsize=(18, 14))

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    linewidths=0.4,
    linecolor='white',
    annot_kws={'size': 7},
    cbar_kws={'shrink': 0.8},
    ax=ax
)

ax.set_title('Heatmap Korelasi — Fitur Numerik Dataset', fontsize=15, fontweight='bold', pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.savefig('correlation_heatmap.png', bbox_inches='tight')
plt.show()
print('\nHeatmap korelasi berhasil ditampilkan.')

In [ ]:
# Korelasi fitur terhadap variabel target 'churn'
churn_corr = corr_matrix['churn'].drop('churn').sort_values(key=abs, ascending=False)

print('=== KORELASI FITUR NUMERIK TERHADAP CHURN (diurutkan dari tertinggi) ===')
churn_corr_df = pd.DataFrame({
    'Fitur'       : churn_corr.index,
    'Korelasi'    : churn_corr.values,
    'Abs Korelasi': churn_corr.abs().values
})
print(churn_corr_df.to_string(index=False))

In [ ]:
# Visualisasi korelasi terhadap Churn
fig, ax = plt.subplots(figsize=(10, max(5, len(churn_corr) * 0.4 + 2)))

colors_bar = ['#E8604C' if v > 0 else '#4C9BE8' for v in churn_corr.values]

bars = ax.barh(
    churn_corr.index,
    churn_corr.values,
    color=colors_bar,
    edgecolor='white',
    height=0.7
)

# Label nilai
for bar, val in zip(bars, churn_corr.values):
    offset = 0.005 if val >= 0 else -0.005
    ha     = 'left' if val >= 0 else 'right'
    ax.text(
        bar.get_width() + offset,
        bar.get_y() + bar.get_height() / 2,
        f'{val:.3f}', va='center', ha=ha, fontsize=8, color='#333333'
    )

ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Koefisien Korelasi Pearson', fontsize=11)
ax.set_title('Korelasi Fitur Numerik terhadap Variabel Target (Churn)',
             fontsize=13, fontweight='bold', pad=12)
ax.invert_yaxis()
ax.spines[['top', 'right']].set_visible(False)

# Legenda warna
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#E8604C', label='Korelasi Positif'),
    Patch(facecolor='#4C9BE8', label='Korelasi Negatif')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=9)

plt.tight_layout()
plt.savefig('churn_correlation_bar.png', bbox_inches='tight')
plt.show()
print('\n Grafik korelasi terhadap Churn berhasil ditampilkan.')

In [ ]:
# Identifikasi hubungan antar variabel yang berpotensi mempengaruhi Churn
threshold = 0.1  # ambang batas minimal korelasi yang dianggap relevan

strong_corr = churn_corr_df[churn_corr_df['Abs Korelasi'] >= threshold].copy()

print(f'=== FITUR DENGAN KORELASI >= {threshold} TERHADAP CHURN ===')
if strong_corr.empty:
    print(f'Tidak ada fitur dengan korelasi >= {threshold}.')
    print('Gunakan threshold lebih kecil atau pertimbangkan analisis non-linear.')
else:
    for _, row in strong_corr.iterrows():
        direction = 'positif ↑' if row['Korelasi'] > 0 else 'negatif ↓'
        print(f"  {row['Fitur']:<35} | r = {row['Korelasi']:+.4f} | {direction}")

print('\n Interpretasi:')
print('  r > 0 → fitur naik, kemungkinan churn naik (risiko lebih tinggi)')
print('  r < 0 → fitur naik, kemungkinan churn turun (lebih loyal)')

---
## Ringkasan Temuan EDA

| Aspek | Temuan |
|---|---|
| **Jumlah Data** | 15.000 records, 30 kolom |
| **Missing Value** | Lihat output Tugas 2 |
| **Keseimbangan Kelas** | Lihat output Tugas 3 (waspadai imbalance) |
| **Fitur Berkorelasi Tinggi dengan Churn** | Lihat output Tugas 4 |

